<a href="https://colab.research.google.com/github/mk654/SML_PG60/blob/main/COMP90051_ProjectGroup60_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **COMP90051 Group Project → Code**

| Project Group 60 |         |
|------------------|---------|
| Lachlan Fox      | 649622  |
| Songhao Guo      | 1542657 |
| Amelia King      | 1175861 |


github → https://github.com/mk654/SML_PG60


In [15]:
# RUN FIRST
from scipy.io import loadmat
import pandas as pd
from pathlib import Path
import numpy as np

import re
import nltk

from scipy import sparse
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer


# Amelia -> load data from github repo
repo_dir = Path("/content/SML_PG60")
processed_dir = repo_dir / "data" / "processed"
if not repo_dir.exists():
    !git clone https://github.com/mk654/SML_PG60 /content/SML_PG60
else:
    print("Repository already exists:", repo_dir)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CATBOOST_TASK_TYPE = "GPU" if torch.cuda.is_available() else "CPU"
CATBOOST_DEVICES = "0" if CATBOOST_TASK_TYPE == "GPU" else None

print("PyTorch device:", DEVICE)
print("CatBoost task type:", CATBOOST_TASK_TYPE)

flu_mat = repo_dir / "data" / "matraw" / "influenza_outbreak_dataset.mat" # access influenza dataset from gitrepo
fludata = loadmat(flu_mat)

c_df = pd.read_csv(repo_dir / "data/covid19_tweets.csv.xz")

# url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/influenza_outbreak_dataset.mat" # old access -> directory = main
#url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/data/influenza_outbreak_dataset.mat" # old access -> directory = main/data


Repository already exists: /content/SML_PG60
PyTorch device: cpu
CatBoost task type: CPU


## preprocessing datasets

### Convert ```influenza_outbreak_dataset.mat``` to .csv
*influenza_outbreak_dataset.mat contains 48 folds (test/train splits). Each fold contains its own X & y train and X & y test:*
<div style="font-size: 0.75em;">

| name     | outer dtype | outer shape | inner type | inner shape |
|----------|-------------|-------------|------------|-------------|
| X train  | object      | (1, 48)     | csc_matrix | (1095, 545) |
| X test   | object      | (1, 48)     | csc_matrix | (485, 545)  |
| y train  | object      | (1, 48)     | ndarray    | (1095, 1)   |
| y test   | object      | (1, 48)     | ndarray    | (485, 1)    |
| locs     | object      | (1, 48)     | ndarray    | (1,)        |
| keywords | object      | (1, 525)    | ndarray    | (1,)        |

</div>

---

For further inspection/analysis, conversion produces 48 separate folders aligning with 48 folds within ```influenza_outbreak_dataset.mat```. Each folder contains:
  1. X_train.csv
  2. X_test.csv
  3. y_train.csv
  4. y_test.csv

Additionally, the conversion also produces:
  1. keywords.csv
  2. locs.csv

In [2]:
# Amelia -> convert influenza_outbreak_dataset.mat to .csv file
flu_out = repo_dir / "data" / "processed" / "flu_csv" / "ak_flucsv"
flu_out.mkdir(parents=True, exist_ok=True)

X_tr = fludata["flu_X_tr"]
X_te = fludata["flu_X_te"]
y_tr = fludata["flu_Y_tr"]
y_te = fludata["flu_Y_te"]

n_folds = X_tr.shape[1]

for i in range(n_folds):
    fold_dir = flu_out / f"fold_{i:02d}"
    fold_dir.mkdir(exist_ok=True)

    Xtr = X_tr[0, i].toarray() # some data stored as sparse matrix, convert to dense
    Xte = X_te[0, i].toarray()
    ytr = y_tr[0, i].ravel()
    yte = y_te[0, i].ravel()

    pd.DataFrame(Xtr).to_csv(fold_dir / "X_train.csv", index=False)
    pd.DataFrame(Xte).to_csv(fold_dir / "X_test.csv", index=False)
    pd.DataFrame(ytr).to_csv(fold_dir / "y_train.csv", index=False)
    pd.DataFrame(yte).to_csv(fold_dir / "y_test.csv", index=False)

keywords = fludata["flu_keywords"]
keywords_list = [str(k[0]) for k in keywords.ravel()]
pd.DataFrame(keywords_list, columns=["keyword"]).to_csv(flu_out / "keywords.csv", index=False)

locs = fludata["flu_locs"]
locs_list = [str(l[0]) for l in locs.ravel()]
pd.DataFrame(locs_list, columns=["location"]).to_csv(flu_out / "locs.csv", index=False)

combines all information from influenza_outbreak_dataset.mat into  ```flu_long.csv``` (~168.6 MB)
- original X matrices within .mat file contain 545 features, however, only 525 named keyword features exist
  * thus ```flu_long.csv``` drops last 20 features within original X matrices
- drops original test/train split from ```influenza_outbreeak_dataset.mat```

In [3]:
# Songhao -> combine all folds into one long-format .csv
combined_out = Path(repo_dir / "data/processed")
combined_out.mkdir(parents=True, exist_ok=True)
#flu_out = Path(flucsv_dir/ "ak_flucsv")
keywords = pd.read_csv(flu_out / "keywords.csv")["keyword"].tolist()
locs = pd.read_csv(flu_out / "locs.csv")["location"].tolist()

all_parts = []

for i, loc in enumerate(locs):
    fold_dir = flu_out / f"fold_{i:02d}"

    X_train = pd.read_csv(fold_dir / "X_train.csv")
    y_train = pd.read_csv(fold_dir / "y_train.csv")
    X_test = pd.read_csv(fold_dir / "X_test.csv")
    y_test = pd.read_csv(fold_dir / "y_test.csv")

    X = pd.concat([X_train, X_test], ignore_index=True)
    y = pd.concat([y_train, y_test], ignore_index=True)
    X= np.asarray(X)
    X = X[:, :len(keywords)] # keep only keyword features
    y = np.asarray(y).reshape(-1).astype(int)

    df_part = pd.DataFrame(X, columns=keywords)
    df_part.insert(0, "location", loc)

    df_part["label"] = y

    all_parts.append(df_part)
df = pd.concat(all_parts, ignore_index=True)
csv_output_path = combined_out / "flu_long.csv"
df.to_csv(csv_output_path, index=False)

print("flu_long shape:", df.shape)
df.head()

flu_long shape: (75840, 527)


,location,flu,swine,stomach,symptoms,virus,bug,strep,season,influenza,...,tests,thinks,ankle,work,hand,complications,children,start,aja,label
0,wyoming,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,wyoming,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,wyoming,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,wyoming,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,wyoming,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


### ```covid19_tweets.csv```
Filter out instances with:
- empty location values
- locations outside of the US

In [4]:
#print(c_df.columns)
#print(c_df["user_location"].head())
import re

c_df["user_location"] = c_df["user_location"].fillna("").astype(str).str.strip().str.lower()
# dictionaries
abbr_to_state = {
    "al": "alabama","ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california","co": "colorado",
    "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon",
    "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina", "sd": "south dakota",
    "tn": "tennessee", "tx": "texas", "ut": "utah", "vt": "vermont", "va": "virginia",
    "wa": "washington", "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia"
}
state_names = set(abbr_to_state.values())

usa_terms = {"usa", "us", "united states", "america", "u.s.", "u.s.a."}

city_to_state = {
    "new york": "new york", "nyc": "new york", "brooklyn": "new york", "manhattan": "new york",
    "los angeles": "california", "san diego": "california", "san francisco": "california", "sacramento": "california", "long beach": "california",
    "houston": "texas", "dallas": "texas", "austin": "texas",
    "miami": "florida", "orlando": "florida",
    "chicago": "illinois",
    "atlanta": "georgia",
    "boston": "massachusetts",
    "las vegas": "nevada",
    "seattle": "washington",
    "new orleans": "louisiana",
    "st louis": "missouri", "saint louis": "missouri",
    "washington dc": "district of columbia"
}

# specific non-us locations/terms to exclude
junk = ["everywhere", "worldwide", "23 countries", "opt-out", "catch me", "the beach",
        "in the vineyard", "available now", "working"]

foreign = ["canada", "uk", "australia", "india", "germany", "france", "italy", "spain", "brazil",
           "argentina", "china", "japan", "south korea", "paris", "london", "vienna", "delhi", "rio",
           "beijing", "nairobi", "joburg"]

bad_ab = {"in", "or", "me", "hi"}

def is_ambig(loc): # multiple locations or terms foreign to usa
    separators = [",", ";", "|", "/", " and ", " & "]
    foreign_count = 0
    for term in foreign:
        if term in loc:
            foreign_count += 1
    if foreign_count >= 1 and any (sep in loc for sep in separators):
        return True
    return False

def is_junk(loc):
    for term in junk:
        if term in loc:
            return True
    return False

#filter
def get_state(loc):
    loc = loc.lower().strip()

    if loc == "":
        return None

    if is_junk(loc):
        return None

    if is_ambig(loc):
        return None

    for state in state_names:
        if state in loc:
            return state

    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return abbr_to_state[token]

    for city in city_to_state:
        if city in loc:
            return city_to_state[city]
    return None

evaluate confidence levels for state disambiguation for filtered c19 twitter data

*millie note: keep? hmm...*

In [5]:
# amelia -> confidence levels for location extraction (high, medium, low, none)
def loc_conf(loc):
    loc = loc.lower().strip()
    for state in state_names:
        if state in loc:
            return "high"
    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return "high"
    for city in city_to_state:
        if city in loc:
            return "medium"
    for term in usa_terms:
        if term in loc:
            return "low"
    return "none"

c_df["state"] = c_df["user_location"].apply(get_state)
c_df["state_confidence"] = c_df["user_location"].apply(loc_conf)
state_cdf = c_df[c_df["state"].notna()].copy()
print(state_cdf[["user_location", "state", "state_confidence"]].head(100))
print(len(state_cdf), "rows with identified US state locations")

#state_cdf[["user_location", "state", "state_confidence"]].to_csv(dir/"data/processed/c19_loc_review.csv", index=False)

             user_location     state state_confidence
1             new york, ny  new york             high
2         pewee valley, ky  kentucky             high
6          gainesville, fl   florida             high
19            florida, usa   florida             high
22       northwest indiana   indiana             high
..                     ...       ...              ...
471     mount prospect, il  illinois             high
473                vermont   vermont             high
474  saint louis, missouri  missouri             high
475           florida, usa   florida             high
476          virginia, usa  virginia             high

[100 rows x 3 columns]
38522 rows with identified US state locations


remove irrelvant features columns:
- ```user_name```
- ```user_description```
- ```user_created```
- ```user_followers```  ⇒ KEEP???
- ```user_friends```    ⇒ KEEP????
- ```user_favourites```
- ```user_verfied```
- ```source```          ⇒ KEEP????
- ```is_retweet```


In [6]:
# amelia -> remove irrelevant feature columns from covid19_tweets.csv
columns_to_keep = ["state", "date", "text", "hashtags"]
cdf_reduced = (c_df[c_df["state"].notna()][columns_to_keep].copy())
#cdf_reduced.to_csv(dir / "data/processed/covid19_twts.csv", index=False)
print(cdf_reduced.head())
print(cdf_reduced.shape)

       state                 date  \
1   new york  2020-07-25 12:27:17   
2   kentucky  2020-07-25 12:27:14   
6    florida  2020-07-25 12:27:03   
19   florida  2020-07-25 12:26:39   
22   indiana  2020-07-25 12:26:31   

                                                 text  \
1   Hey @Yankees @YankeesPR and @MLB - wouldn't it...   
2   @diane3443 @wdunlap @realDonaldTrump Trump nev...   
6   How #COVID19 Will Change Work in General (and ...   
19  COVID Update: The infection rate in Florida is...   
22  @JimBnntt Your image doesn't list a source, bu...   

                     hashtags  
1                         NaN  
2                 ['COVID19']  
6   ['COVID19', 'Recruiting']  
19                        NaN  
22                        NaN  
(38522, 4)


Create label froom presence of ```covid19``` hashtag

In [7]:
# Lachlan ->
def has_covid19_hashtag(hashtags):
    hashtags = "" if pd.isna(hashtags) else str(hashtags).lower()
    return int("covid19" in hashtags or "covid_19" in hashtags or "covid-19" in hashtags)

cdf_reduced["label"] = cdf_reduced["hashtags"].apply(has_covid19_hashtag)

"""
def hashtag_check(hashtags):
  lower = hashtags.lower()
  targets = ['covid19', 'coronavirus']
  if any(item in lower for item in targets):
    return 1
  else:
    return 0

hashtag_series = covid_df['hashtags'].copy()
hashtag_series = hashtag_series.fillna("")

covid_vector_df['label'] = hashtag_series.apply(hashtag_check)
covid_vector_df.head()
"""

'\ndef hashtag_check(hashtags):\n  lower = hashtags.lower()\n  targets = [\'covid19\', \'coronavirus\']\n  if any(item in lower for item in targets):\n    return 1\n  else:\n    return 0\n\nhashtag_series = covid_df[\'hashtags\'].copy()\nhashtag_series = hashtag_series.fillna("")\n\ncovid_vector_df[\'label\'] = hashtag_series.apply(hashtag_check)\ncovid_vector_df.head()\n'

clean tweet text by removing:
- urls
- hashtags
- mentins
- punctuation
- extra whitespace

Add hashtag label.

In [8]:
# Amelia -> clean tweet text (remove urls, mentions, hashtags, punctuation, extra whitespace)
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s\-]", "", text)
    #text = re.sub(r"[^a-z0-9\s\-]", "", text) # preserves numbers... do we want this?
    text = re.sub(r"\s+", " ", text).strip()
    return text

def comb_tt(row):
    text = "" if pd.isna(row["text"]) else str(row["text"])
    hashtags = "" if pd.isna(row["hashtags"]) else str(row["hashtags"])
    return (text + " " + hashtags)

cdf_reduced["clean_text"] = cdf_reduced["text"].apply(clean_tweet)
print(cdf_reduced[["text", "clean_text"]].head())

                                                 text  \
1   Hey @Yankees @YankeesPR and @MLB - wouldn't it...   
2   @diane3443 @wdunlap @realDonaldTrump Trump nev...   
6   How #COVID19 Will Change Work in General (and ...   
19  COVID Update: The infection rate in Florida is...   
22  @JimBnntt Your image doesn't list a source, bu...   

                                           clean_text  
1   hey and - wouldnt it have made more sense to h...  
2   trump never once claimed covid was a hoax we a...  
6   how covid will change work in general and recr...  
19  covid update the infection rate in florida is ...  
22  your image doesnt list a source but id be care...  


#### process ```covid19_tweets.csv``` for first covid tweet dataset
convert tweet text to dictionary using ```flu_long.csv``` (from influenza_outbreak_dataset.mat); apply
    - this creates the covid dataset 1, wherein

In [16]:
# # Lachlan -> vectorise tweets based on flu keywords
# flu_data = Path(repo_dir / "data/processed/flu_long.csv")
# flu_df= pd.read_csv(flu_data)
# flu_columns = flu_df.columns.tolist()
# columns_to_drop = ['location', 'label']
# flu_keywords = [item for item in flu_columns if item not in columns_to_drop]

# def vectorise_tweet(tweet):
#   #text = tweet.lower()
#   counts = {}
#   for keyword in flu_keywords:
#     kw_clean = str(keyword).lower().strip().replace("_", " ")
#     pattern = r"(?<![a-z0-9])" + re.escape(kw_clean) + r"(?![a-z0-9])"
#     counts[keyword] = len(re.findall(pattern, tweet))
#   return counts

# # Amelia -> apply
# comb_txt = cdf_reduced.apply(comb_tt, axis=1)
# covid_vecs = comb_txt.apply(vectorise_tweet)
# covid_vecs_df = pd.DataFrame(covid_vecs.tolist())

# cdf_reduced = cdf_reduced.rename(columns={"state": "location"})
# c19_twts_d1  = pd.concat([cdf_reduced[["location"]].reset_index(drop=True), covid_vecs_df.reset_index(drop=True),
#                          cdf_reduced[["label"]].reset_index(drop=True)], axis=1)
# c19_twts_d1.to_csv(processed_dir / "c19_twts_d1.csv", index=False)
# print(c19_twts_d1.head())
# print(c19_twts_d1.shape)
# print(c19_twts_d1["label"].value_counts())


# Lachlan -> vectorise tweets based on flu keywords
# Faster version: use CountVectorizer with flu keyword vocabulary

output_path = processed_dir / "c19_twts_d1.csv"

if output_path.exists():
    c19_twts_d1 = pd.read_csv(output_path)
    print("Loaded existing c19_twts_d1.csv")
    print(c19_twts_d1.head())
    print(c19_twts_d1.shape)
    print(c19_twts_d1["label"].value_counts())

else:
    flu_data = processed_dir / "flu_long.csv"
    flu_df = pd.read_csv(flu_data)

    columns_to_drop = ["location", "label", "split", "time_index"]
    flu_keywords = [col for col in flu_df.columns if col not in columns_to_drop]

    # Clean flu keywords in the same way as the original regex code
    clean_to_original = {}
    for keyword in flu_keywords:
        clean_keyword = str(keyword).lower().strip().replace("_", " ")
        if clean_keyword != "" and clean_keyword not in clean_to_original:
            clean_to_original[clean_keyword] = keyword

    clean_keywords = list(clean_to_original.keys())
    max_ngram = max(len(keyword.split()) for keyword in clean_keywords)

    vocabulary = {keyword: idx for idx, keyword in enumerate(clean_keywords)}

    comb_txt = cdf_reduced.apply(comb_tt, axis=1).astype(str).str.lower()

    vectorizer = TfidfVectorizer(
        vocabulary=vocabulary,
        use_idf=False,
        norm=None,
        lowercase=False,
        ngram_range=(1, max_ngram),
    )

    X_counts = vectorizer.fit_transform(comb_txt)

    covid_vecs_df = pd.DataFrame(
        X_counts.toarray(),
        columns=[clean_to_original[keyword] for keyword in clean_keywords],
    )

    # Make sure COVID feature columns exactly follow flu_long.csv columns
    covid_vecs_df = covid_vecs_df.reindex(columns=flu_keywords, fill_value=0)

    cdf_reduced = cdf_reduced.rename(columns={"state": "location"})

    c19_twts_d1 = pd.concat(
        [
            cdf_reduced[["location"]].reset_index(drop=True),
            covid_vecs_df.reset_index(drop=True),
            cdf_reduced[["label"]].reset_index(drop=True),
        ],
        axis=1,
    )

    c19_twts_d1.to_csv(output_path, index=False)

    print(c19_twts_d1.head())
    print(c19_twts_d1.shape)
    print(c19_twts_d1["label"].value_counts())

   location  flu  swine  stomach  symptoms  virus  bug  strep  season  \
0  new york  0.0    0.0      0.0       0.0    0.0  0.0    0.0     0.0   
1  kentucky  0.0    0.0      0.0       0.0    0.0  0.0    0.0     0.0   
2   florida  0.0    0.0      0.0       0.0    0.0  0.0    0.0     0.0   
3   florida  0.0    0.0      0.0       0.0    0.0  0.0    0.0     0.0   
4   indiana  0.0    0.0      0.0       0.0    0.0  0.0    0.0     0.0   

   influenza  ...  tests  thinks  ankle  work  hand  complications  children  \
0        0.0  ...    0.0     0.0    0.0   0.0   0.0            0.0       0.0   
1        0.0  ...    0.0     0.0    0.0   0.0   0.0            0.0       0.0   
2        0.0  ...    0.0     0.0    0.0   1.0   0.0            0.0       0.0   
3        0.0  ...    0.0     0.0    0.0   0.0   0.0            0.0       0.0   
4        0.0  ...    0.0     0.0    0.0   0.0   0.0            0.0       0.0   

   start  aja  label  
0    0.0  0.0      0  
1    0.0  0.0      1  
2    0.0  0

#### process ```covid19_tweets.csv``` for second covid tweet dataset
Lemmatise text

In [17]:
# Amelia -> Lemmatise
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatiser = WordNetLemmatizer()
def lemmatise_tweet(text):
    tokens = text.split()
    #tokens = re.findall(r"\b\w+\b", text.lower())
    lemmas = [lemmatiser.lemmatize(token) for token in tokens]
    return " ".join(lemmas)
cdf_reduced["lemmatised_text"] = cdf_reduced["clean_text"].apply(lemmatise_tweet)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1,2), min_df=5)
X_covid = tfidf.fit_transform(cdf_reduced["clean_text"])

covid_keywords = tfidf.get_feature_names_out()
c_tfidf_df = pd.DataFrame(X_covid.toarray(), columns=covid_keywords)

covid19_twts = pd.concat([cdf_reduced.reset_index(drop=True), c_tfidf_df.reset_index(drop=True)], axis=1) #need to double check this output!
covid19_twts= covid19_twts.rename(columns={"state":"location"})
covid19_twts.to_csv(processed_dir / "c19_twts_d2.csv", index=False)

### Processed data summary
1. flu_long.csv
  - contains
2. c19_twts_d1.csv
   - keywords from flu_long.csv
3. c19_twts_d2.csv
  - keywords from covid19_tweets.csv


In [ ]:

# Amelia -> pipeline of three datasets
# processed_dir = repo_dir / "data" / "processed"
# #datasets = ["flu_long.csv", "c19_twts_d1.csv", "c19_twts_d2.csv"]
# def load_model_data(filename):
#     df = pd.read_csv(processed_dir / filename)
#     drop_cols = ["location", "state", "date", "text", "hashtags", "clean_text","lemmatised_text", "stemmed_text", "split", "time_index"] # drop meta cols if present
#     existing_drop_cols = [col for col in drop_cols if col in df.columns]
#     y = df["label"].astype(int).values
#     X = df.drop(columns=existing_drop_cols + ["label"]).values.astype(float)
#     return X, y

# keep the same shared feature construction of for all three main models
def build_feature_table(filename):
    """
    Load a processed dataset and build the same non-temporal constructed features
    used by Logistic Regression, Bayesian Logistic Regression, and CatBoost.

    For the main experiment, use filename="flu_long.csv".
    COVID files should be treated as optional external weak-label analysis, not
    as the main nested-CV task.
    """
    df = pd.read_csv(processed_dir / filename)

    target_col = "label"
    drop_cols = [
        "location", "state", "date", "text", "hashtags",
        "clean_text", "lemmatised_text", "stemmed_text",
        "split", "time_index"
    ]
    existing_drop_cols = [col for col in drop_cols if col in df.columns]

    feature_cols = [
        col for col in df.columns
        if col not in existing_drop_cols + [target_col]
    ]

    X_raw = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    y = df[target_col].astype(int).to_numpy()

    # Non-temporal feature construction, shared with CatBoost
    X = X_raw.copy()
    X["keyword_total_intensity"] = X_raw.sum(axis=1)
    X["active_keyword_count"] = (X_raw > 0).sum(axis=1)
    X["nonzero_keyword_ratio"] = X["active_keyword_count"] / max(len(feature_cols), 1)
    X["max_keyword_value"] = X_raw.max(axis=1)
    X["mean_keyword_value"] = X_raw.mean(axis=1)
    X["std_keyword_value"] = X_raw.std(axis=1)
    X["log_total_intensity"] = np.log1p(X["keyword_total_intensity"])

    return X, y


def load_model_data(filename):
    """
    Return NumPy arrays for PyTorch models.
    """
    X_df, y = build_feature_table(filename)
    X_np = X_df.to_numpy(dtype=float)
    return X_np, y


## Model training

---



In [ ]:
#Lachlan -> parameter metrics
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

### Logistic Regression

In [ ]:
#Lachlan -> basic logistic regression model
class LogisticRegressionModel(nn.Module): #From tute
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

def train_model(
    model_class,
    input_dim,
    criterion_fn,
    lr,
    momentum,
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    seed=42,
    weight_decay=0.0,
):
    torch.manual_seed(seed)

    model = model_class(input_dim).to(DEVICE)

    # Adam does not use the momentum argument directly.
    # The argument is kept for compatibility with earlier calls.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    Xt = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(DEVICE)

    dataset = torch.utils.data.TensorDataset(Xt, yt)
    train_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    for epoch in range(epochs):
        model.train()

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion_fn(predictions, batch_y)
            loss.backward()
            optimizer.step()

    return model

# Songhao -> stratified folds from scratch
def make_stratified_folds(y, n_folds, seed=42):
  rng = np.random.default_rng(seed)
  y = np.asarray(y).astype(int)

  folds = [[] for _ in range(n_folds)]

  for label in np.unique(y):
      label_indices = np.where(y == label)[0]
      rng.shuffle(label_indices)

      for i, idx in enumerate(label_indices):
          folds[i % n_folds].append(idx)

  return [np.array(fold, dtype=int) for fold in folds]

# Amelia -> implement like sklearn import
def stratified_split_indices(X, y, n_folds, seed=42):
  """uses Songhao's make_stratified_folds function to generate (train_idx, val_idx)
  similar to StratifiedKFold.split(X,y)"""
  val_folds= make_stratified_folds(y, n_folds, seed)
  all_indices= np.arange(len(y))
  for val_idx in val_folds:
    train_idx=np.setdiff1d(all_indices, val_idx)
    yield train_idx, val_idx


In [ ]:
# Lachlan -> Logistic Regression with nested cross-validation

def standardise_train_eval(X_train, X_eval):
    """
    Standardise features using only the training fold statistics.
    This prevents information from the validation/test fold leaking into training.
    """
    train_mean = X_train.mean(axis=0)
    train_std = X_train.std(axis=0)
    train_std[train_std == 0] = 1

    X_train_std = (X_train - train_mean) / train_std
    X_eval_std = (X_eval - train_mean) / train_std

    return X_train_std, X_eval_std


def run_logistic_nested_cv(
    X,
    y,
    outer_folds=10,
    inner_folds=3,
    weight_decay_grid=[1e-5, 1e-4, 1e-3],
    lr=0.001,
    seed=42,
    epochs=50,
):
    """
    Same structure as CatBoost:
    - outer 10-fold stratified CV for performance estimation
    - inner 3-fold stratified CV for hyperparameter tuning
    - tune weight_decay as the Logistic Regression regularisation strength
    """
    criterion = torch.nn.BCEWithLogitsLoss()

    outer_results = []
    tuning_results = []

    outer_splits = stratified_split_indices(
        X,
        y,
        n_folds=outer_folds,
        seed=seed,
    )

    for outer_fold_id, (outer_train_idx, outer_test_idx) in enumerate(outer_splits, start=1):
        print(f"Logistic Regression outer fold {outer_fold_id}/{outer_folds}")

        X_outer_train = X[outer_train_idx]
        y_outer_train = y[outer_train_idx]
        X_outer_test = X[outer_test_idx]
        y_outer_test = y[outer_test_idx]

        best_weight_decay = None
        best_inner_f1 = -1.0

        # Inner CV for tuning
        for weight_decay in weight_decay_grid:
            inner_scores = []

            inner_splits = stratified_split_indices(
                X_outer_train,
                y_outer_train,
                n_folds=inner_folds,
                seed=seed + outer_fold_id,
            )

            for inner_fold_id, (inner_train_idx, inner_val_idx) in enumerate(inner_splits, start=1):
                X_inner_train = X_outer_train[inner_train_idx]
                y_inner_train = y_outer_train[inner_train_idx]
                X_inner_val = X_outer_train[inner_val_idx]
                y_inner_val = y_outer_train[inner_val_idx]

                X_inner_train, X_inner_val = standardise_train_eval(
                    X_inner_train,
                    X_inner_val,
                )

                model = train_model(
                    model_class=LogisticRegressionModel,
                    input_dim=X_inner_train.shape[1],
                    criterion_fn=criterion,
                    lr=lr,
                    momentum=0.0,
                    X_train=X_inner_train,
                    y_train=y_inner_train,
                    epochs=epochs,
                    seed=seed + outer_fold_id + inner_fold_id,
                    weight_decay=weight_decay,
                )

                model.eval()
                X_val_tensor = torch.tensor(X_inner_val, dtype=torch.float32).to(DEVICE)

                with torch.no_grad():
                    logits = model(X_val_tensor)
                    y_val_pred = (torch.sigmoid(logits) >= 0.5).float().cpu().numpy().flatten()

                val_metrics = compute_metrics(y_inner_val, y_val_pred)
                inner_scores.append(val_metrics["f1"])

            mean_inner_f1 = float(np.mean(inner_scores))
            std_inner_f1 = float(np.std(inner_scores, ddof=1))

            tuning_results.append({
                "outer_fold": outer_fold_id,
                "weight_decay": weight_decay,
                "mean_inner_f1": mean_inner_f1,
                "std_inner_f1": std_inner_f1,
            })

            if mean_inner_f1 > best_inner_f1:
                best_inner_f1 = mean_inner_f1
                best_weight_decay = weight_decay

        outer_mean = X_outer_train.mean(axis=0)
        outer_std = X_outer_train.std(axis=0)
        outer_std[outer_std == 0] = 1

        X_outer_train_std = (X_outer_train - outer_mean) / outer_std
        X_outer_test_std = (X_outer_test - outer_mean) / outer_std

        final_model = train_model(
            model_class=LogisticRegressionModel,
            input_dim=X_outer_train_std.shape[1],
            criterion_fn=criterion,
            lr=lr,
            momentum=0.0,
            X_train=X_outer_train_std,
            y_train=y_outer_train,
            epochs=epochs,
            seed=seed + 100 + outer_fold_id,
            weight_decay=best_weight_decay,
        )

        final_model.eval()
        X_test_tensor = torch.tensor(X_outer_test_std, dtype=torch.float32).to(DEVICE)

        with torch.no_grad():
            logits = final_model(X_test_tensor)
            y_test_pred = (torch.sigmoid(logits) >= 0.5).float().cpu().numpy().flatten()

        test_metrics = compute_metrics(y_outer_test, y_test_pred)
        test_metrics["outer_fold"] = outer_fold_id
        test_metrics["best_weight_decay"] = best_weight_decay
        test_metrics["best_inner_f1"] = best_inner_f1

        outer_results.append(test_metrics)
    logistic_results_df = pd.DataFrame(outer_results)
    logistic_tuning_df = pd.DataFrame(tuning_results)

    logistic_summary_df = pd.DataFrame([
        {
            "metric": metric,
            "mean": float(logistic_results_df[metric].mean()),
            "std": float(logistic_results_df[metric].std(ddof=1)),
        }
        for metric in ["accuracy", "precision", "recall", "f1"]
    ])

    return logistic_results_df, logistic_tuning_df, logistic_summary_df

In [ ]:
# Lachlan -> run baseline Logistic Regression on the main influenza task only

# Main supervised experiment:
# flu_long.csv + same constructed features + outer 10-fold / inner 3-fold nested CV
X_np, y_np = load_model_data("flu_long.csv")

logistic_results_df, logistic_tuning_df, logistic_summary_df = run_logistic_nested_cv(
    X=X_np,
    y=y_np,
    outer_folds=10,
    inner_folds=3,
    weight_decay_grid=[1e-5, 1e-4, 1e-3],
    lr=0.001,
    seed=42,
    epochs=50,
)

display(logistic_results_df)
display(logistic_tuning_df)
display(logistic_summary_df)


### Bayesian Regression

In [ ]:
#Lachlan -> Bayesian regression model
class BayesianLogisticRegressionModel(nn.Module):
  def __init__(self, input_dim):
    super(BayesianLogisticRegressionModel,self).__init__()
    self.input_dim = input_dim
    #Need weights w and bias b with mean mu and variance proxy rho
    #Initially assume weights and bias have mean 0
    self.w_mu = nn.Parameter(torch.randn(input_dim, 1) * 0.001) #Small randomization of initial means
    self.w_rho = nn.Parameter(torch.zeros(input_dim, 1))
    self.b_mu = nn.Parameter(torch.zeros(1))
    self.b_rho = nn.Parameter(torch.zeros(1))
    #Note that the actual standard deviation sigma = ln(1 + exp(rho)) This is to ensure that sigma is positive definite

  def forward(self, X):
    #Change to actual standard deviation
    w_sigma = torch.log(1 + torch.exp(self.w_rho))
    b_sigma = torch.log(1 + torch.exp(self.b_rho))

    #Generate posterior weight and bias distributions
    self.w_posterior = torch.distributions.Normal(self.w_mu, w_sigma)
    self.b_posterior = torch.distributions.Normal(self.b_mu, b_sigma)

    #Sample from weight and bias distributions
    w = self.w_posterior.rsample()
    b = self.b_posterior.rsample()

    #Calculate logits
    logits = X @ w + b
    return logits

  def kl_divergence(self):
    # Define prior distribution around w, b like ~Normal(0, 1)
    w_prior = torch.distributions.Normal(torch.zeros_like(self.w_mu), torch.ones_like(self.w_rho))
    b_prior = torch.distributions.Normal(torch.zeros_like(self.b_mu), torch.ones_like(self.b_rho))

    #Calculate Kullback-Leibler divergence between prior and posterior distributions
    w_kl = torch.distributions.kl.kl_divergence(self.w_posterior, w_prior).sum()
    b_kl = torch.distributions.kl.kl_divergence(self.b_posterior, b_prior).sum()

    return w_kl + b_kl

def elbo_loss(model, y_pred, y_batch, n_batches, kl_weight=1.0):
  bce = torch.nn.BCEWithLogitsLoss(reduction = "sum") #Sum reduction needed to maintain scale with KL divergence
  neg_log_likelihood = bce(y_pred, y_batch)
  kl = model.kl_divergence()
  loss = neg_log_likelihood + kl_weight * kl / n_batches
  return loss


In [ ]:
#Lachlan -> train Bayesian regression model
def train_bayes_model(
    model_class,
    input_dim,
    lr,
    momentum,
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    seed=42,
    kl_weight=1.0,
):
    torch.manual_seed(seed)
    model = model_class(input_dim).to(DEVICE)


    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    Xt = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(DEVICE)

    dataset = torch.utils.data.TensorDataset(Xt, yt)
    train_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    n_batches = len(train_loader)

    for epoch in range(epochs):
      model.train()
      for batch_X, batch_y in train_loader:
          optimizer.zero_grad()
          predictions = model(batch_X)
          loss = elbo_loss(
              model,
              predictions,
              batch_y,
              n_batches,
              kl_weight=kl_weight,
          )
          loss.backward()
          optimizer.step()

    return model


In [ ]:
# Lachlan -> Bayesian Logistic Regression with nested cross-validation

def predict_bayes_model(model, X_eval, n_samples=100):
    model.eval()
    X_eval_tensor = torch.tensor(X_eval, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        samples = []

        for _ in range(n_samples):
            logits = model(X_eval_tensor)
            probs = torch.sigmoid(logits)
            samples.append(probs)

        mean_probs = torch.stack(samples).mean(dim=0)
        y_pred = (mean_probs >= 0.5).float().cpu().numpy().flatten()

    return y_pred


def run_bayesian_nested_cv(
    X,
    y,
    outer_folds=10,
    inner_folds=3,
    kl_weight_grid=[1e-4, 1e-3, 1e-2], #hyperparameter tuning
    lr=0.001,
    seed=42,
    epochs=50,
    n_samples_inner=30,
    n_samples_outer=100,
):
    """
    Same structure as CatBoost:
    - outer 10-fold stratified CV for performance estimation
    - inner 3-fold stratified CV for hyperparameter tuning
    - tune kl_weight as the Bayesian regularisation strength
    """
    outer_results = []
    tuning_results = []

    outer_splits = stratified_split_indices(
        X,
        y,
        n_folds=outer_folds,
        seed=seed,
    )

    for outer_fold_id, (outer_train_idx, outer_test_idx) in enumerate(outer_splits, start=1):
        print(f"Bayesian Logistic Regression outer fold {outer_fold_id}/{outer_folds}")

        X_outer_train = X[outer_train_idx]
        y_outer_train = y[outer_train_idx]
        X_outer_test = X[outer_test_idx]
        y_outer_test = y[outer_test_idx]

        best_kl_weight = None
        best_inner_f1 = -1.0

        # Inner CV for tuning
        for kl_weight in kl_weight_grid:
            inner_scores = []

            inner_splits = stratified_split_indices(
                X_outer_train,
                y_outer_train,
                n_folds=inner_folds,
                seed=seed + outer_fold_id,
            )

            for inner_fold_id, (inner_train_idx, inner_val_idx) in enumerate(inner_splits, start=1):
                X_inner_train = X_outer_train[inner_train_idx]
                y_inner_train = y_outer_train[inner_train_idx]
                X_inner_val = X_outer_train[inner_val_idx]
                y_inner_val = y_outer_train[inner_val_idx]

                X_inner_train, X_inner_val = standardise_train_eval(
                    X_inner_train,
                    X_inner_val,
                )

                model = train_bayes_model(
                    model_class=BayesianLogisticRegressionModel,
                    input_dim=X_inner_train.shape[1],
                    lr=lr,
                    momentum=0.0,
                    X_train=X_inner_train,
                    y_train=y_inner_train,
                    epochs=epochs,
                    seed=seed + outer_fold_id + inner_fold_id,
                    kl_weight=kl_weight,
                )

                y_val_pred = predict_bayes_model(
                    model,
                    X_inner_val,
                    n_samples=n_samples_inner,
                )

                val_metrics = compute_metrics(y_inner_val, y_val_pred)
                inner_scores.append(val_metrics["f1"])

            mean_inner_f1 = float(np.mean(inner_scores))
            std_inner_f1 = float(np.std(inner_scores, ddof=1))

            tuning_results.append({
                "outer_fold": outer_fold_id,
                "kl_weight": kl_weight,
                "mean_inner_f1": mean_inner_f1,
                "std_inner_f1": std_inner_f1,
            })

            if mean_inner_f1 > best_inner_f1:
                best_inner_f1 = mean_inner_f1
                best_kl_weight = kl_weight

        outer_mean = X_outer_train.mean(axis=0)
        outer_std = X_outer_train.std(axis=0)
        outer_std[outer_std == 0] = 1

        X_outer_train_std = (X_outer_train - outer_mean) / outer_std
        X_outer_test_std = (X_outer_test - outer_mean) / outer_std

        final_model = train_bayes_model(
            model_class=BayesianLogisticRegressionModel,
            input_dim=X_outer_train_std.shape[1],
            lr=lr,
            momentum=0.0,
            X_train=X_outer_train_std,
            y_train=y_outer_train,
            epochs=epochs,
            seed=seed + 100 + outer_fold_id,
            kl_weight=best_kl_weight,
        )

        y_test_pred = predict_bayes_model(
            final_model,
            X_outer_test_std,
            n_samples=n_samples_outer,
        )

        test_metrics = compute_metrics(y_outer_test, y_test_pred)
        test_metrics["outer_fold"] = outer_fold_id
        test_metrics["best_kl_weight"] = best_kl_weight
        test_metrics["best_inner_f1"] = best_inner_f1

        outer_results.append(test_metrics)
    bayesian_results_df = pd.DataFrame(outer_results)
    bayesian_tuning_df = pd.DataFrame(tuning_results)

    bayesian_summary_df = pd.DataFrame([
        {
            "metric": metric,
            "mean": float(bayesian_results_df[metric].mean()),
            "std": float(bayesian_results_df[metric].std(ddof=1)),
        }
        for metric in ["accuracy", "precision", "recall", "f1"]
    ])

    return bayesian_results_df, bayesian_tuning_df, bayesian_summary_df

In [ ]:
# Lachlan -> run Bayesian Logistic Regression on the main influenza task only

# Main supervised experiment:
# flu_long.csv + same constructed features + outer 10-fold / inner 3-fold nested CV
X_np, y_np = load_model_data("flu_long.csv")

bayesian_results_df, bayesian_tuning_df, bayesian_summary_df = run_bayesian_nested_cv(
    X=X_np,
    y=y_np,
    outer_folds=10,
    inner_folds=3,
    kl_weight_grid=[1e-4, 1e-3, 1e-2],
    lr=0.001,
    seed=42,
    epochs=50,
    n_samples_inner=30,
    n_samples_outer=100,
)

display(bayesian_results_df)
display(bayesian_tuning_df)
display(bayesian_summary_df)


###  CatBoost
This section adds CatBoost as the complex model.

In [ ]:
# Songhao -> CatBoost model
!pip install catboost
from catboost import CatBoostClassifier

In [ ]:
# Songhao -> Prepare features and labels for CatBoost

# Use exactly the same constructed feature table as Logistic Regression and Bayesian Logistic Regression.
X, y = build_feature_table("flu_long.csv")

catboost_feature_columns = X.columns.tolist()

print("Final feature shape:", X.shape)
print("Label distribution:")
print(pd.Series(y).value_counts())


In [ ]:
#  Songhao -> Metrics and stratified k-fold cross-validation from scratch

# commented out for now; code for this is above in logistic regression cell
# should probably put common functions for cross-val and so on at the start of the notebook for easy access
"""def make_stratified_folds(y, n_folds, seed=42):
  rng = np.random.default_rng(seed)
  y = np.asarray(y).astype(int)

  folds = [[] for _ in range(n_folds)]

  for label in np.unique(y):
      label_indices = np.where(y == label)[0]
      rng.shuffle(label_indices)

      for i, idx in enumerate(label_indices):
          folds[i % n_folds].append(idx)

  return [np.array(fold, dtype=int) for fold in folds] """


def get_train_indices(n_samples, test_indices):
    all_indices = np.arange(n_samples)
    return np.setdiff1d(all_indices, test_indices)

def summarise_results(results_df):
    summary_rows = []

    for metric in ["accuracy", "precision", "recall", "f1"]:
        values = results_df[metric].to_numpy(dtype=float)

        summary_rows.append({
            "metric": metric,
            "mean": values.mean(),
            "std": values.std(ddof=1),
        })

    return pd.DataFrame(summary_rows)

In [ ]:
# Songhao -> CatBoost with nested cross-validation
def train_catboost(X_train, y_train, depth, iterations=200, learning_rate=0.05, seed=42):
    model = CatBoostClassifier(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        loss_function="Logloss",
        eval_metric="F1",
        auto_class_weights="Balanced",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        task_type=CATBOOST_TASK_TYPE,
        devices="0" if CATBOOST_TASK_TYPE == "GPU" else None,
    )

    model.fit(X_train, y_train)
    return model

def tune_catboost_depth(X_train_outer, y_train_outer, depth_grid, inner_folds=3, seed=42):
    inner_fold_indices = make_stratified_folds(
        y_train_outer,
        n_folds=inner_folds,
        seed=seed,
    )

    tuning_results = []

    for depth in depth_grid:
        inner_scores = []

        for inner_fold_id, val_idx in enumerate(inner_fold_indices):
            train_idx = get_train_indices(len(y_train_outer), val_idx)

            X_inner_train = X_train_outer.iloc[train_idx]
            y_inner_train = y_train_outer[train_idx]
            X_inner_val = X_train_outer.iloc[val_idx]
            y_inner_val = y_train_outer[val_idx]

            model = train_catboost(
                X_inner_train,
                y_inner_train,
                depth=depth,
                iterations=120,
                learning_rate=0.05,
                seed=seed + inner_fold_id,
            )

            y_val_pred = model.predict(X_inner_val).astype(int)
            val_metrics = compute_metrics(y_inner_val, y_val_pred)
            inner_scores.append(val_metrics["f1"])

        tuning_results.append({
            "depth": depth,
            "mean_inner_f1": float(np.mean(inner_scores)),
            "std_inner_f1": float(np.std(inner_scores, ddof=1)),
        })

    tuning_df = pd.DataFrame(tuning_results)
    best_depth = int(
        tuning_df.sort_values("mean_inner_f1", ascending=False).iloc[0]["depth"]
    )

    return best_depth, tuning_df

outer_folds = make_stratified_folds(y, n_folds=10, seed=42)
depth_grid = [10, 12, 14]

catboost_outer_results = []
catboost_tuning_logs = []

for outer_fold_id, test_idx in enumerate(outer_folds, start=1):
    print(f"Running outer fold {outer_fold_id}/10")

    train_idx = get_train_indices(len(y), test_idx)

    X_outer_train = X.iloc[train_idx].reset_index(drop=True)
    y_outer_train = y[train_idx]
    X_outer_test = X.iloc[test_idx].reset_index(drop=True)
    y_outer_test = y[test_idx]

    best_depth, tuning_df = tune_catboost_depth(
        X_outer_train,
        y_outer_train,
        depth_grid=depth_grid,
        inner_folds=3,
        seed=100 + outer_fold_id,
    )

    tuning_df["outer_fold"] = outer_fold_id
    catboost_tuning_logs.append(tuning_df)

    final_model = train_catboost(
        X_outer_train,
        y_outer_train,
        depth=best_depth,
        iterations=200,
        learning_rate=0.05,
        seed=200 + outer_fold_id,
    )

    y_test_pred = final_model.predict(X_outer_test).astype(int)
    test_metrics = compute_metrics(y_outer_test, y_test_pred)

    test_metrics["outer_fold"] = outer_fold_id
    test_metrics["best_depth"] = best_depth

    catboost_outer_results.append(test_metrics)

catboost_results_df = pd.DataFrame(catboost_outer_results)
catboost_tuning_df = pd.concat(catboost_tuning_logs, ignore_index=True)
catboost_summary_df = summarise_results(catboost_results_df)

display(catboost_results_df)
display(catboost_summary_df)


In [ ]:
# Save all main nested-CV results and final model artefacts

results_dir = processed_dir / "main_nested_cv_results"
results_dir.mkdir(parents=True, exist_ok=True)

model_dir = processed_dir / "final_models"
model_dir.mkdir(parents=True, exist_ok=True)

# Save nested-CV result tables
logistic_results_df.to_csv(results_dir / "logistic_outer_cv_results.csv", index=False)
logistic_tuning_df.to_csv(results_dir / "logistic_inner_tuning_results.csv", index=False)
logistic_summary_df.to_csv(results_dir / "logistic_summary.csv", index=False)

bayesian_results_df.to_csv(results_dir / "bayesian_outer_cv_results.csv", index=False)
bayesian_tuning_df.to_csv(results_dir / "bayesian_inner_tuning_results.csv", index=False)
bayesian_summary_df.to_csv(results_dir / "bayesian_summary.csv", index=False)

catboost_results_df.to_csv(results_dir / "catboost_outer_cv_results.csv", index=False)
catboost_tuning_df.to_csv(results_dir / "catboost_inner_tuning_results.csv", index=False)
catboost_summary_df.to_csv(results_dir / "catboost_summary.csv", index=False)

main_summary_df = pd.concat(
    [
        logistic_summary_df.assign(model="Logistic Regression"),
        bayesian_summary_df.assign(model="Bayesian Logistic Regression"),
        catboost_summary_df.assign(model="CatBoost"),
    ],
    ignore_index=True,
)

main_summary_df.to_csv(results_dir / "main_summary_all_models.csv", index=False)

# Select final hyperparameters in the same style as CatBoost:
# use the value most frequently selected across the outer folds.
best_weight_decay_overall = float(logistic_results_df["best_weight_decay"].mode().iloc[0])
best_kl_weight_overall = float(bayesian_results_df["best_kl_weight"].mode().iloc[0])
best_depth_overall = int(catboost_results_df["best_depth"].mode().iloc[0])

selected_hyperparameters_df = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "hyperparameter": "weight_decay",
        "selected_value": best_weight_decay_overall,
    },
    {
        "model": "Bayesian Logistic Regression",
        "hyperparameter": "kl_weight",
        "selected_value": best_kl_weight_overall,
    },
    {
        "model": "CatBoost",
        "hyperparameter": "depth",
        "selected_value": best_depth_overall,
    },
])

selected_hyperparameters_df.to_csv(
    results_dir / "selected_final_hyperparameters.csv",
    index=False,
)

# Train and save each final model once.
# The weak-label test below only loads these saved models and does not retrain them.
X_flu_np, y_flu = load_model_data("flu_long.csv")

flu_mean = X_flu_np.mean(axis=0)
flu_std = X_flu_np.std(axis=0)
flu_std[flu_std == 0] = 1

X_flu_std = (X_flu_np - flu_mean) / flu_std

criterion = torch.nn.BCEWithLogitsLoss()

final_logistic_model = train_model(
    model_class=LogisticRegressionModel,
    input_dim=X_flu_std.shape[1],
    criterion_fn=criterion,
    lr=0.001,
    momentum=0.0,
    X_train=X_flu_std,
    y_train=y_flu,
    epochs=50,
    seed=999,
    weight_decay=best_weight_decay_overall,
)

final_bayesian_model = train_bayes_model(
    model_class=BayesianLogisticRegressionModel,
    input_dim=X_flu_std.shape[1],
    lr=0.001,
    momentum=0.0,
    X_train=X_flu_std,
    y_train=y_flu,
    epochs=50,
    seed=999,
    kl_weight=best_kl_weight_overall,
)

X_catboost_final, y_catboost_final = build_feature_table("flu_long.csv")

final_catboost_model = train_catboost(
    X_train=X_catboost_final,
    y_train=y_catboost_final,
    depth=best_depth_overall,
    iterations=200,
    learning_rate=0.05,
    seed=999,
)

torch.save(
    {
        "model_state_dict": final_logistic_model.state_dict(),
        "input_dim": X_flu_std.shape[1],
        "best_weight_decay": best_weight_decay_overall,
        "mean": flu_mean,
        "std": flu_std,
    },
    model_dir / "final_logistic_model.pt",
)

torch.save(
    {
        "model_state_dict": final_bayesian_model.state_dict(),
        "input_dim": X_flu_std.shape[1],
        "best_kl_weight": best_kl_weight_overall,
        "mean": flu_mean,
        "std": flu_std,
    },
    model_dir / "final_bayesian_model.pt",
)

final_catboost_model.save_model(str(model_dir / "final_catboost_model.cbm"))

print("Saved nested-CV results to:", results_dir)
print("Saved final model artefacts to:", model_dir)

display(main_summary_df)
display(selected_hyperparameters_df)


## Weak_Label_Test(domain transfer performance)

In [ ]:
# Weak-label domain transfer test using saved final models only

model_dir = processed_dir / "final_models"
results_dir = processed_dir / "main_nested_cv_results"

logistic_model_path = model_dir / "final_logistic_model.pt"
bayesian_model_path = model_dir / "final_bayesian_model.pt"
catboost_model_path = model_dir / "final_catboost_model.cbm"

missing_paths = [
    str(path)
    for path in [logistic_model_path, bayesian_model_path, catboost_model_path]
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Saved final model file(s) missing. Run the previous save/final-model cell first:\n"
        + "\n".join(missing_paths)
    )

# Load saved Logistic Regression model
logistic_ckpt = torch.load(logistic_model_path, map_location=DEVICE)
final_logistic_model = LogisticRegressionModel(logistic_ckpt["input_dim"]).to(DEVICE)
final_logistic_model.load_state_dict(logistic_ckpt["model_state_dict"])
final_logistic_model.eval()

# Load saved Bayesian Logistic Regression model
bayesian_ckpt = torch.load(bayesian_model_path, map_location=DEVICE)
final_bayesian_model = BayesianLogisticRegressionModel(bayesian_ckpt["input_dim"]).to(DEVICE)
final_bayesian_model.load_state_dict(bayesian_ckpt["model_state_dict"])
final_bayesian_model.eval()

# Load saved CatBoost model
final_catboost_model = CatBoostClassifier()
final_catboost_model.load_model(str(catboost_model_path))

selected_hyperparameters_df = pd.read_csv(results_dir / "selected_final_hyperparameters.csv")

# Use c19_twts_d1.csv because it is vectorised using the influenza keyword feature space.
X_covid_df, y_covid = build_feature_table("c19_twts_d1.csv")

# Align COVID feature columns to the influenza feature columns used during final training.
X_flu_df, _ = build_feature_table("flu_long.csv")
X_covid_df = X_covid_df.reindex(columns=X_flu_df.columns, fill_value=0)
X_covid_np = X_covid_df.to_numpy(dtype=float)

# Logistic Regression prediction
X_covid_logistic_std = (X_covid_np - logistic_ckpt["mean"]) / logistic_ckpt["std"]

with torch.no_grad():
    X_covid_tensor = torch.tensor(X_covid_logistic_std, dtype=torch.float32).to(DEVICE)
    logistic_logits = final_logistic_model(X_covid_tensor)
    logistic_pred = (torch.sigmoid(logistic_logits) >= 0.5).float().cpu().numpy().flatten()

logistic_weak_metrics = compute_metrics(y_covid, logistic_pred)

# Bayesian Logistic Regression prediction
X_covid_bayesian_std = (X_covid_np - bayesian_ckpt["mean"]) / bayesian_ckpt["std"]

bayesian_pred = predict_bayes_model(
    final_bayesian_model,
    X_covid_bayesian_std,
    n_samples=100,
)

bayesian_weak_metrics = compute_metrics(y_covid, bayesian_pred)

# CatBoost prediction
catboost_pred = final_catboost_model.predict(X_covid_df).astype(int)
catboost_weak_metrics = compute_metrics(y_covid, catboost_pred)

def get_selected_value(model_name):
    return selected_hyperparameters_df.loc[
        selected_hyperparameters_df["model"] == model_name,
        "selected_value",
    ].iloc[0]

weak_label_results_df = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "final_hyperparameter": "weight_decay",
        "final_value": float(get_selected_value("Logistic Regression")),
        **logistic_weak_metrics,
    },
    {
        "model": "Bayesian Logistic Regression",
        "final_hyperparameter": "kl_weight",
        "final_value": float(get_selected_value("Bayesian Logistic Regression")),
        **bayesian_weak_metrics,
    },
    {
        "model": "CatBoost",
        "final_hyperparameter": "depth",
        "final_value": int(get_selected_value("CatBoost")),
        **catboost_weak_metrics,
    },
])

weak_label_results_df.to_csv(
    results_dir / "covid19_weak_label_external_test_saved_final_models.csv",
    index=False,
)

display(weak_label_results_df)

print(
    "Saved weak-label test results to:",
    results_dir / "covid19_weak_label_external_test_saved_final_models.csv",
)


## Misc. Are we keeping this stuff?

In [ ]:
#Lachlan -> keyword counting
flu_df = pd.read_csv(processed_dir / "flu_long.csv")
flu_columns = flu_df.columns.tolist()
columns_to_drop = ['location', 'split', 'time_index', 'label']
flu_keywords = [item for item in flu_columns if item not in columns_to_drop]

keyword_totals = {}
for keyword in flu_keywords:
  total = flu_df[keyword].sum()
  keyword_totals[keyword] = total

sorted_keywords = sorted(keyword_totals, key = keyword_totals.get)

In [ ]:
#Lachlan -> reduction by dropping columns
num_keys = len(sorted_keywords)
x = 100 #Number of desired keywords
num_to_drop = num_keys - x
columns_to_drop = sorted_keywords[:num_to_drop]
reduced_flu_df = flu_df.drop(columns = columns_to_drop)
reduced_flu_df.head(5)


In [ ]:
#Lachlan -> autoencoder

class FeatureAutoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()

        hidden_dim = max(256, bottleneck_dim * 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def encode_data(X, new_dim, epochs = 50, batch_size = 32, lr = 0.001):
  #Convert to tensor
  Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
  X_dim = Xt.shape[1]

  dataset = torch.utils.data.TensorDataset(Xt)
  data_loader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True)

  autoencoder_model = FeatureAutoencoder(input_dim=X_dim, bottleneck_dim=new_dim).to(DEVICE)
  criterion = nn.MSELoss()
  #Use Adam for best performance
  optimizer = torch.optim.Adam(autoencoder_model.parameters(), lr = lr)

  autoencoder_model.train()
  for i in range(epochs):
    for (batch_x,) in data_loader:
      reconstructed_data = autoencoder_model(batch_x)
      loss = criterion(reconstructed_data, batch_x)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  autoencoder = autoencoder_model.encoder
  autoencoder.eval()
  with torch.no_grad():
    reduced_X = autoencoder(Xt)

  new_X_np = reduced_X.detach().cpu().numpy()

  return new_X_np


In [ ]:
#Lachlan -> Determine optimal autoencoding
# This exploratory function is not part of the final three-model main experiment.
# It is kept for reference only. The old cross_val() function has been replaced
# by nested CV for the main Logistic Regression experiment.

#X and y must be numpy arrays
#X must be 2D

def opt_autoencode(k_folds, step_size, X, y, epochs=50, batch_size=32, lr=0.001, momentum=1e-2):
  raise NotImplementedError(
      "Autoencoder selection is not part of the final main experiment. "
      "Use run_logistic_nested_cv(), run_bayesian_nested_cv(), and the CatBoost nested CV section instead."
  )


In [ ]:
#Lachlan -> Autoencode X

#from sklearn.preprocessing import StandardScaler
#df = pd.read_csv("flu_long.csv")
#df.drop(columns = ['location'], inplace = True) #Basic version
#df['label'] = df['label'].astype(int) #Cast entries as ints

#scaler = StandardScaler()
#X_np = scaler.fit_transform((df.drop(columns = ['label'])).values)
#y_np = df['label'].values
#k_folds = 10
#step_size = 50
#epochs = 30
#batch_size = 32
#lr = 0.001
#momentum = 1e-2


#X_reduced = opt_autoencode(k_folds, step_size, X_np, y_np, epochs, batch_size, lr, momentum)
#print(X_reduced.shape)


In [ ]:
# Final CatBoost training moved above

# The final CatBoost model is now trained and saved together with Logistic Regression
# and Bayesian Logistic Regression in the main results/final-model artefact cell.
# Do not rerun final CatBoost training here to avoid duplicate computation.
